In [1]:
import cv2
import numpy as np

import os
import glob

from typing import List, Tuple

In [2]:
def prepare_data(data_path):
    """ 
    Prepares list of file paths for real and fake signatures, from a specific folder.

    if the subdirectory name ends with 'g', the files are considered fake signatures;
    otherwise, they are considered real signatures.

    Example:
        real_signatures_paths, fake_signatures_paths = prepare_data('../data_path')
    """
    fake_signatures_paths = []
    real_signatures_paths = []

    for per in os.listdir(data_path):
        for data in glob.glob(os.path.join(data_path, per, '*.png')):
            if per[-1] == 'g':
                fake_signatures_paths.append(data)
            else:
                real_signatures_paths.append(data)
    return real_signatures_paths, fake_signatures_paths

In [3]:
def process_signature(image):
    """ 
    Given an image path, this function reads and transfrom the image. 
    It reads, resize, binarize and morphed the image. 

    Example:
        processed_imgage = process_signature(path/to/image)
    """
    image = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (635, 145))
    _, image = cv2.threshold(image, 200, 255, cv2.THRESH_BINARY)
    kernel = np.ones((2,2), np.uint8)
    image = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel, iterations=2)
    return image

In [4]:
def calculate_signature_area(image):
    """
    count number of black pixels, in a white background and black image.
    https://stackoverflow.com/questions/32590932/count-number-of-black-pixels-in-an-image-in-python-with-opencv
    """
    signature_area = cv2.countNonZero(image)
    return (image.shape[0]*image.shape[1]) - signature_area


def calculate_slant_angle(image):
    """
    Binarize image and find the largest contour in the image, fit a line to the contour.
    it then calculate the angle and normilize it.

    example: 
        calculate_slant_angle(path/to/image)
    """
    _, binary = cv2.threshold(image, 200, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    largest_contour = max(contours, key=cv2.contourArea)
    [x1, y1, x2, y2] = cv2.fitLine(largest_contour, cv2.DIST_L2, 0, 0.01, 0.01)
    angle = np.arctan2(y2 - y1, x2 - x1) * (180 / np.pi)
    if angle < 0:
        angle += 180
    return angle.item()


def calculate_stroke_thickness(image):
    """
    Stroke thickness can be approximated by measuring the thickness of horizontal and vertical lines in the signature.
    
    example: 
        stroke_thickness(processed_image2)
    """
    kernel = np.ones((2,2), np.uint8)
    dilate = cv2.dilate(image, kernel, iterations=2)
    dilate_result = (dilate.shape[0]*dilate.shape[1]) - cv2.countNonZero(image) 
    stroke_thickness = dilate_result - cv2.countNonZero(dilate)
    return stroke_thickness


def calculate_pixel_distribution(image):
    """ 
    Calculate pixel density distribution of an image. 

    Example:
        mean, std_density = calculate_pixel_distribution(image)
    """
    grid_size = 10
    height, width = image.shape

    pixel_density = []
    for i in range(0, height, grid_size):
        for j in range(0, width, grid_size):
            grid = image[i:min(i + grid_size, height), j:min(j + grid_size, width)]
            density = (grid_size * grid_size - cv2.countNonZero(grid)) / (grid.shape[0] * grid.shape[1])
            pixel_density.append(density)

    return np.mean(pixel_density), np.std(pixel_density)

In [5]:
def extract_features(image):
    """ 
    a dictionary to hold the features of a single image

    Example:
        imgage_features = extract_features(image)
    """
    features = {
        "signature_area": calculate_signature_area(image),
        "slant_angle": calculate_slant_angle(image),
        "stroke_thickness": calculate_stroke_thickness(image),
        "pixel_density": calculate_pixel_distribution(image)
    }
    return features

In [6]:
 def calculate_average_profile(signatures):
    """ 
    function to compute mean and standard deviation for features across a set of signatures.
    extract features for each signature and store in the respective lists.

    avg_profile: calculate the mean and standard deviation of each feature.

    Returns:
        dictionary.

    Example:
        genuine_profile = calculate_average_profile(paths/to/signatures/dir)
    """
    areas, angles, thicknesses, density_means, density_stds = [], [], [], [], []

    for image in signatures:
        image = process_signature(image)
        features = extract_features(image)
        
        areas.append(features["signature_area"])
        angles.append(features["slant_angle"])
        thicknesses.append(features["stroke_thickness"])
        density_means.append(features["pixel_density"][0])
        density_stds.append(features["pixel_density"][1])

   
    avg_profile = {
        "signature_area": (np.mean(areas), np.std(areas)),
        "slant_angle": (np.mean(angles), np.std(angles)),
        "stroke_thickness": (np.mean(thicknesses), np.std(thicknesses)),
        "pixel_density": (np.mean(density_means), np.std(density_means), 
                          np.mean(density_stds), np.std(density_stds))
    }
    
    return avg_profile

In [7]:
def compare_with_profile(features, avg_profile, tolerance=3):
    """ 
    function to compare a single signature feature, with the average profile built from other signatures 
    """
    match = True
    for key, avg_values in avg_profile.items():
        if key == "pixel_density":
            # For pixel density, handle the four values (mean density mean, std deviation of density mean, mean of std dev, std deviation of std dev)
            mean_density, std_density, mean_stddev, stddev_stddev = avg_values

            # Check both the mean density and the standard deviation of density
            if (abs(features[key][0] - mean_density) > tolerance * std_density or
                abs(features[key][1] - mean_stddev) > tolerance * stddev_stddev):
                match = False
                print(f"{key} difference too high: {features[key]}")
        else:
            # For other scalar features, each has mean and standard deviation
            mean, std_dev = avg_values
            
            # Check if the feature value is within tolerance * standard deviation from the mean
            if abs(features[key] - mean) > tolerance * std_dev:
                match = False
                print(f"{key} difference too high: {features[key]}")
    return match


In [9]:
for i in range(len(real_signatures_paths)):
    
    new_signature = process_signature(real_signatures_paths[i])
    new_features = extract_features(new_signature)

    # Compare the new signature with the average genuine profile
    is_genuine = compare_with_profile(new_features, genuine_profile)
    if is_genuine:
        print("The new signature is similar to the genuine profile.")
    else:
        print("The new signature differs from the genuine profile (potential forgery).")

The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine pr

In [10]:
for i in range(len(fake_signatures_paths)):
    new_signature = process_signature(fake_signatures_paths[i])
    new_features = extract_features(new_signature)

    # Compare the new signature with the average genuine profile
    is_genuine = compare_with_profile(new_features, genuine_profile)
    if is_genuine:
        print("The new signature is similar to the genuine profile.")
    else:
        print("The new signature differs from the genuine profile (potential forgery).")


The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
The new signature is similar to the genuine profile.
signature_area difference too high: 5521
slant_angle difference too high: 52.70005416870117
stroke_thickness difference too high: -84645
pixel_density difference too high: (np.float64(0.1419895833333333), np.float64(0.3098163096494863))
The new signature differs from the genuine profile (potential forgery).
signature_area difference too high: 5102
slant_angle difference too high: 46.62739181518555
stroke_thickness difference too high: -85129
pixel_density difference too high: (np.float64(0.13733333333333334), np.float64(0.3058502038943611))
The new signature differs from the genuine profile (potential forgery).
stroke_thickness difference too high: -83089
pixel_density difference too high: (np.float64(0.14928125), np.float64(0.3083393086062347))
The new signature differs from t

In [11]:
# this test image is not included in the statitical profile creation
test_path = '../signature_recognition/002_01.PNG'

new_signature = process_signature(test_path)
new_features = extract_features(new_signature)

is_genuine = compare_with_profile(new_features, genuine_profile)
if is_genuine:
    print("The new signature is similar to the genuine profile.")
else:
    print("The new signature differs from the genuine profile (potential forgery).")

The new signature is similar to the genuine profile.
